In [ ]:
!pip install ta

In [ ]:
# ============================================================
# CELL 1 — Setup: Clone repo & install dependencies
# ============================================================
import os, subprocess, sys

REPO_URL   = "https://github.com/whatsoever025/Multimodal-Cryptocurrency-Market-Sentiment-Forecasting.git"
REPO_DIR   = "/kaggle/working/crypto"
WANDB_KEY  = ""   # ← dán W&B key nếu muốn dùng
HF_TOKEN   = ""   # ← dán Hugging Face token nếu muốn đẩy dataset lên HF

if not os.path.exists(REPO_DIR):
    # Clone và checkout nhánh main (nhánh chính đang dùng cho mọi thay đổi)
    subprocess.run(["git", "clone", "-b", "main", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"✓ Working dir: {os.getcwd()}")

# Cài đặt các thư viện bổ sung còn thiếu trên Kaggle
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers", "datasets", "wandb", "huggingface_hub",
    "scikit-learn", "timm", "Pillow",
], check=True)
print("✓ Dependencies installed")

# Đăng nhập W&B (tùy chọn)
if WANDB_KEY:
    import wandb
    wandb.login(key=WANDB_KEY)
    print("✓ W&B logged in")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("⚠ W&B disabled (no key provided)")

# Đăng nhập Hugging Face (tùy chọn)
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("✓ HuggingFace logged in")


In [ ]:
# ============================================================
# CELL 2 — Extract features for both BTC and ETH
#   Runs extraction once per image backbone in IMAGE_BACKBONES.
#   "vit" writes image_embeddings.pt (thesis's primary embeddings).
#   "clip" writes image_embeddings_clip.pt (diagnostic-only alternative,
#   see thesis Section 5.3 "Future Work" / reviewer question 6).
#   text/tabular/target extraction is only ever done once (skipped on the
#   second backbone pass since those files already exist and --force is
#   NOT passed) — only the image embeddings differ between passes.
# ============================================================
FEATURES_DIR = "/kaggle/working/features_v5"
os.makedirs(FEATURES_DIR, exist_ok=True)

IMAGE_BACKBONES = ["vit", "clip"]   # ← bỏ "vit" nếu bạn đã có sẵn image_embeddings.pt từ trước

for backbone in IMAGE_BACKBONES:
    print("=" * 60)
    print(f"Extracting features (BTC & ETH) — image backbone: {backbone}")
    print("Estimated time with GPU: ~25-30 minutes (vit) / ~10-15 minutes (clip, text+tabular+target skipped)")
    print("=" * 60)

    result = subprocess.run([
        sys.executable, "-m", "src.training.extract_features",
        "--asset", "MULTI",
        "--output_dir", FEATURES_DIR,
        "--image-backbone", backbone,
    ], capture_output=False)

    if result.returncode != 0:
        raise RuntimeError(f"Feature extraction failed for backbone={backbone}! Check logs above.")

# Xác minh kích thước các file .pt vừa trích xuất
import torch, json
print("\n" + "=" * 60)
print("Feature shapes verification:")
print("=" * 60)

IMAGE_FILES = {"vit": "image_embeddings.pt", "clip": "image_embeddings_clip.pt"}

for asset in ["BTC", "ETH"]:
    asset_dir = os.path.join(FEATURES_DIR, asset)
    if not os.path.exists(asset_dir):
        print(f"❌ ERROR: Missing asset directory {asset_dir}!")
        continue

    print(f"\nChecking {asset} features:")
    shapes = {}
    fnames = ["text_embeddings.pt", "tabular_features_extended.pt", "target_scores.pt"]
    fnames += [IMAGE_FILES[b] for b in IMAGE_BACKBONES]
    for fname in fnames:
        fpath = os.path.join(asset_dir, fname)
        if os.path.exists(fpath):
            t = torch.load(fpath, map_location="cpu")
            shapes[fname] = t.shape[0]
            print(f"  {fname}: {tuple(t.shape)}")
        else:
            print(f"  ❌ Missing: {fname}")

    meta_path = os.path.join(asset_dir, "split_metadata.json")
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        print(f"  split_metadata: {meta}")

    # Đảm bảo số dòng thẳng hàng nhau giữa các file của asset
    if len(shapes) == len(fnames) and len(set(shapes.values())) == 1:
        N = list(shapes.values())[0]
        print(f"  ✅ All {N} rows aligned across {len(fnames)} {asset} feature files.")
    else:
        print(f"  ❌ MISALIGNMENT OR MISSING FILES in {asset}: {shapes}")


In [ ]:
# ============================================================
# CELL 4 — Push fresh features to Kaggle dataset
#   Run this only AFTER extraction is verified to work correctly.
# ============================================================
PUSH_TO_KAGGLE = True   # ← set True to upload

if PUSH_TO_KAGGLE:
    KAGGLE_USERNAME = "namkhanhng"
    KAGGLE_KEY      = "c8d35b256cd9417262f02d4970eb7a82"

    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

    # 1. Copy dataset-metadata.json từ repository nếu có
    import shutil, json as _json
    meta_src = os.path.join(REPO_DIR, "data", "features", "dataset-metadata.json")
    meta_dst = os.path.join(FEATURES_DIR, "dataset-metadata.json")
    if os.path.exists(meta_src):
        shutil.copy(meta_src, meta_dst)
        print("✓ Copied metadata from repository")
    else:
        # 2. Đảm bảo luôn tự động sinh file metadata nếu chưa tồn tại
        dataset_metadata = {
            "title": "Multimodal Crypto Features v5",
            "id": f"{KAGGLE_USERNAME}/multimodal-crypto-features-v5",
            "licenses": [{"name": "CC0-1.0"}]
        }
        with open(meta_dst, "w") as f:
            _json.dump(dataset_metadata, f, indent=4)
        print("✓ Created default dataset-metadata.json")

    subprocess.run(["pip", "install", "-q", "kaggle"], check=True)

    os.chdir(FEATURES_DIR)
    
    # 3. Sử dụng tham số --dir-mode zip để nén và tải lên các thư mục con BTC & ETH
    try:
        subprocess.run([
            "kaggle", "datasets", "version",
            "-p", ".",
            "-m", "v5-separate-assets: BTC and ETH in separate subfolders (aligned)",
            "--dir-mode", "zip"
        ], check=True)
        print("✅ Pushed fresh features to Kaggle dataset as a new version!")
    except subprocess.CalledProcessError:
        print("Dataset might not exist yet on Kaggle, attempting to create new dataset...")
        subprocess.run([
            "kaggle", "datasets", "create", 
            "-p", ".", 
            "-u",
            "--dir-mode", "zip"
        ], check=True)
        print("✅ Created new Kaggle dataset and uploaded features!")
        
    os.chdir(REPO_DIR)
